In [70]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

RAW_FOLDER = PROJECT_ROOT / "data" / "raw"
INTERIM_FOLDER = PROJECT_ROOT / "data" / "interim"
NOTEBOOK_FOLDER = PROJECT_ROOT / "notebooks"
REPORT_FOLDER = PROJECT_ROOT / "reports"

print("Project folder:", PROJECT_ROOT)
print("Raw folder exists:", RAW_FOLDER.exists())
print("Interim folder exists:", INTERIM_FOLDER.exists())
print("Notebooks folder exists:", NOTEBOOK_FOLDER.exists())
print("Reports folder exists:", REPORT_FOLDER.exists())

Project folder: C:\Users\disha\ecommerce-delivery-logistics-analysis
Raw folder exists: True
Interim folder exists: True
Notebooks folder exists: True
Reports folder exists: True


In [71]:
import pandas as pd

csv_files = list(RAW_FOLDER.glob("*.csv"))

print("Total CSV files:", len(csv_files))
print()

for file in sorted(csv_files):
    df_temp = pd.read_csv(file, nrows=5)
    print(
        file.name,
        "| Columns:", len(df_temp.columns),
        "| Column names:", list(df_temp.columns)
    )

Total CSV files: 9

olist_customers_dataset.csv | Columns: 5 | Column names: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
olist_geolocation_dataset.csv | Columns: 5 | Column names: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
olist_order_items_dataset.csv | Columns: 7 | Column names: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
olist_order_payments_dataset.csv | Columns: 5 | Column names: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
olist_order_reviews_dataset.csv | Columns: 7 | Column names: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
olist_orders_dataset.csv | Columns: 8 | Column names: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'or

In [72]:
import pandas as pd

audit_results = []

for file in sorted(RAW_FOLDER.glob("*.csv")):
    df = pd.read_csv(file)

    audit_results.append({
        "Dataset": file.name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Cells": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    })

audit_df = pd.DataFrame(audit_results)

audit_df

,Dataset,Rows,Columns,Missing Cells,Duplicate Rows
0,olist_customers_dataset.csv,99441,5,0,0
1,olist_geolocation_dataset.csv,1000163,5,0,261831
2,olist_order_items_dataset.csv,112650,7,0,0
3,olist_order_payments_dataset.csv,103886,5,0,0
4,olist_order_reviews_dataset.csv,99224,7,145903,0
5,olist_orders_dataset.csv,99441,8,4908,0
6,olist_products_dataset.csv,32951,9,2448,0
7,olist_sellers_dataset.csv,3095,4,0,0
8,product_category_name_translation.csv,71,2,0,0


In [73]:
audit_df = pd.DataFrame(audit_results)

audit_df = audit_df[
    ["Dataset", "Rows", "Columns", "Missing Cells", "Duplicate Rows"]
]

display(audit_df.to_string(index=False))

'                              Dataset    Rows  Columns  Missing Cells  Duplicate Rows\n          olist_customers_dataset.csv   99441        5              0               0\n        olist_geolocation_dataset.csv 1000163        5              0          261831\n        olist_order_items_dataset.csv  112650        7              0               0\n     olist_order_payments_dataset.csv  103886        5              0               0\n      olist_order_reviews_dataset.csv   99224        7         145903               0\n             olist_orders_dataset.csv   99441        8           4908               0\n           olist_products_dataset.csv   32951        9           2448               0\n            olist_sellers_dataset.csv    3095        4              0               0\nproduct_category_name_translation.csv      71        2              0               0'

In [74]:
customers = pd.read_csv(RAW_FOLDER / "olist_customers_dataset.csv")

print("Total rows:", len(customers))
print("Duplicate customer_id:", customers["customer_id"].duplicated().sum())
print("Missing values per column:")
print(customers.isna().sum())

Total rows: 99441
Duplicate customer_id: 0
Missing values per column:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [75]:
customers_clean = customers.copy()

customers_clean.to_csv(
    INTERIM_FOLDER / "customers_clean.csv",
    index=False
)

print("Customers cleaned file saved successfully!")
print("Rows:", len(customers_clean))
print("Missing cells:", customers_clean.isna().sum().sum())

Customers cleaned file saved successfully!
Rows: 99441
Missing cells: 0


In [76]:
geo = pd.read_csv(RAW_FOLDER / "olist_geolocation_dataset.csv")

print("Total rows:", len(geo))
print("\nMissing values per column:")
print(geo.isna().sum())

print("\nExact duplicate rows:", geo.duplicated().sum())

print(
    "Unique ZIP codes:",
    geo["geolocation_zip_code_prefix"].nunique()
)

Total rows: 1000163

Missing values per column:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Exact duplicate rows: 261831
Unique ZIP codes: 19015


In [77]:
geo[geo.duplicated(keep=False)].sort_values(
    by=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
).head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP


In [78]:
geo_clean = geo.drop_duplicates().copy()

geo_clean.to_csv(
    INTERIM_FOLDER / "geolocation_clean.csv",
    index=False
)

print("Original rows:", len(geo))
print("Cleaned rows:", len(geo_clean))
print("Duplicates remaining:", geo_clean.duplicated().sum())
print("Missing cells:", geo_clean.isna().sum().sum())

Original rows: 1000163
Cleaned rows: 738332
Duplicates remaining: 0
Missing cells: 0


In [79]:
order_items = pd.read_csv(
    RAW_FOLDER / "olist_order_items_dataset.csv"
)

print("Total rows:", len(order_items))
print("\nMissing values:")
print(order_items.isna().sum())

print("\nExact duplicate rows:", order_items.duplicated().sum())

print(
    "\nDuplicate order-item combinations:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Total rows: 112650

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Exact duplicate rows: 0

Duplicate order-item combinations: 0


In [80]:
order_items_check = order_items.copy()

order_items_check["shipping_limit_date"] = pd.to_datetime(
    order_items_check["shipping_limit_date"],
    errors="coerce"
)

print(
    "Invalid shipping dates:",
    order_items_check["shipping_limit_date"].isna().sum()
)

print("Negative prices:", (order_items_check["price"] < 0).sum())
print("Negative freight values:", (order_items_check["freight_value"] < 0).sum())

print("\nColumn data types:")
print(order_items_check.dtypes)


Invalid shipping dates: 0
Negative prices: 0
Negative freight values: 0

Column data types:
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [81]:
order_items_clean = order_items_check.copy()

order_items_clean.to_csv(
    INTERIM_FOLDER / "order_items_clean.csv",
    index=False
)

print("Order Items file saved successfully!")
print("Rows:", len(order_items_clean))
print("Missing cells:", order_items_clean.isna().sum().sum())
print(
    "Duplicate rows:",
    order_items_clean.duplicated().sum()
)

Order Items file saved successfully!
Rows: 112650
Missing cells: 0
Duplicate rows: 0


In [82]:
payments = pd.read_csv(
    RAW_FOLDER / "olist_order_payments_dataset.csv"
)

print("Total rows:", len(payments))

print("\nMissing values:")
print(payments.isna().sum())

print("\nExact duplicate rows:", payments.duplicated().sum())

print(
    "Duplicate order-payment combinations:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

print("\nPayment types:")
print(payments["payment_type"].value_counts(dropna=False))

Total rows: 103886

Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Exact duplicate rows: 0
Duplicate order-payment combinations: 0

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [83]:
print("Payment types marked not_defined:")
display(payments[payments["payment_type"] == "not_defined"])

print("Negative payment values:", (payments["payment_value"] < 0).sum())
print("Zero payment values:", (payments["payment_value"] == 0).sum())
print("Zero or negative installments:",
      (payments["payment_installments"] <= 0).sum())

print("\nData types:")
print(payments.dtypes)

Payment types marked not_defined:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


Negative payment values: 0
Zero payment values: 9
Zero or negative installments: 2

Data types:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object


In [84]:
unusual_payments = payments[
    (payments["payment_value"] == 0) |
    (payments["payment_installments"] <= 0)
]

display(unusual_payments.sort_values(
    by=["payment_value", "payment_installments"]
))

print("Total unusual records:", len(unusual_payments))

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


Total unusual records: 11


In [85]:
print(payments.loc[[46982, 79014]].to_string())

print("\nMissing values in these records:")
print(payments.loc[[46982, 79014]].isna())

                               order_id  payment_sequential payment_type  payment_installments  payment_value
46982  744bade1fcf9ff3f31d860ace076d422                   2  credit_card                     0          58.69
79014  1a57108394169c0b47d8f876acc9ba2d                   2  credit_card                     0         129.94

Missing values in these records:
       order_id  payment_sequential  payment_type  payment_installments  \
46982     False               False         False                 False   
79014     False               False         False                 False   

       payment_value  
46982          False  
79014          False  


In [86]:
payments_clean = payments.copy()

payments_clean.to_csv(
    INTERIM_FOLDER / "payments_clean.csv",
    index=False
)

print("Payments file saved successfully!")
print("Rows:", len(payments_clean))
print("Missing cells:", payments_clean.isna().sum().sum())
print("Duplicate rows:", payments_clean.duplicated().sum())

Payments file saved successfully!
Rows: 103886
Missing cells: 0
Duplicate rows: 0


In [87]:
reviews = pd.read_csv(
    RAW_FOLDER / "olist_order_reviews_dataset.csv"
)

print("Total rows:", len(reviews))
print("\nMissing values per column:")
print(reviews.isna().sum())

print("\nExact duplicate rows:", reviews.duplicated().sum())

print("\nColumn data types:")
print(reviews.dtypes)

Total rows: 99224

Missing values per column:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Exact duplicate rows: 0

Column data types:
review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object


In [88]:
print("Review score distribution:")
print(reviews["review_score"].value_counts().sort_index())

print("\nInvalid review scores:")
print(
    (~reviews["review_score"].between(1, 5)).sum()
)

print("\nDuplicate review IDs:")
print(reviews["review_id"].duplicated().sum())

print("\nDuplicate order IDs:")
print(reviews["order_id"].duplicated().sum())

for col in ["review_creation_date", "review_answer_timestamp"]:
    parsed_dates = pd.to_datetime(reviews[col], errors="coerce")
    print(f"Invalid dates in {col}:", parsed_dates.isna().sum())

Review score distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Invalid review scores:
0

Duplicate review IDs:
814

Duplicate order IDs:
551
Invalid dates in review_creation_date: 0
Invalid dates in review_answer_timestamp: 0


In [89]:
duplicate_reviews = reviews[
    reviews.duplicated(subset=["review_id"], keep=False)
].sort_values("review_id")

display(duplicate_reviews.head(15))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [91]:
duplicate_id_summary = (
    reviews[reviews["review_id"].duplicated(keep=False)]
    .groupby("review_id")
    .agg(
        Total_Rows=("order_id", "size"),
        Unique_Orders=("order_id", "nunique"),
        Unique_Scores=("review_score", "nunique"),
        Unique_Titles=(
            "review_comment_title",
            lambda x: x.nunique(dropna=False)
        ),
        Unique_Messages=(
            "review_comment_message",
            lambda x: x.nunique(dropna=False)
        ),
        Unique_Creation_Dates=("review_creation_date", "nunique"),
        Unique_Answer_Timestamps=("review_answer_timestamp", "nunique")
    )
    .reset_index()
)

display(duplicate_id_summary.head(10))

print("Repeated review_id groups:", len(duplicate_id_summary))

,review_id,Total_Rows,Unique_Orders,Unique_Scores,Unique_Titles,Unique_Messages,Unique_Creation_Dates,Unique_Answer_Timestamps
0,00130cbe1f9d422698c812ed8ded1919,2,2,1,1,1,1,1
1,0115633a9c298b6a98bcbe4eee75345f,2,2,1,1,1,1,1
2,0174caf0ee5964646040cd94e15ac95e,2,2,1,1,1,1,1
3,017808d29fd1f942d97e50184dfb4c13,2,2,1,1,1,1,1
4,0254bd905dc677a6078990aad3331a36,2,2,1,1,1,1,1
5,0288d42bef3dfe36930740c9588a570f,2,2,1,1,1,1,1
6,02aa7f5f75e964e3c7efa59a1f515281,2,2,1,1,1,1,1
7,034528cb00b6bc981847acafbdf2ae0b,2,2,1,1,1,1,1
8,03a6a25db577d0689440933055111897,2,2,1,1,1,1,1
9,0467560f511c516ddaa54a60edb0c291,2,2,1,1,1,1,1


Repeated review_id groups: 789


In [92]:
reviews_clean = reviews.copy()

reviews_clean["review_comment_title"] = (
    reviews_clean["review_comment_title"].fillna("")
)

reviews_clean["review_comment_message"] = (
    reviews_clean["review_comment_message"].fillna("")
)

for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col], errors="coerce"
    )

reviews_clean.to_csv(
    INTERIM_FOLDER / "reviews_clean.csv",
    index=False
)

print("File saved successfully!")
print("Rows:", len(reviews_clean))
print("Missing cells:", reviews_clean.isna().sum().sum())
print("Duplicate rows:", reviews_clean.duplicated().sum())
print("File exists:", (INTERIM_FOLDER / "reviews_clean.csv").exists())

File saved successfully!
Rows: 99224
Missing cells: 0
Duplicate rows: 0
File exists: True


In [93]:
orders = pd.read_csv(
    RAW_FOLDER / "olist_orders_dataset.csv"
)

print("Total rows:", len(orders))

print("\nMissing values per column:")
print(orders.isna().sum())

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

print("\nMissing dates by order status:")
display(
    orders.groupby("order_status")[date_columns]
    .apply(lambda group: group.isna().sum())
)

Total rows: 99441

Missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Missing dates by order status:


,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [94]:
orders_clean = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Conversion se pehle existing missing values count karo
original_missing = orders_clean[date_columns].isna().sum()

# Dates ko datetime format mein convert karo
for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

# Conversion ke baad missing values count karo
converted_missing = orders_clean[date_columns].isna().sum()

print("New invalid dates introduced during conversion:")
print((converted_missing - original_missing).clip(lower=0))

print("\nDelivered orders with missing customer delivery date:")

display(
    orders_clean.loc[
        (orders_clean["order_status"] == "delivered")
        & orders_clean["order_delivered_customer_date"].isna(),
        ["order_id", "order_status", "order_purchase_timestamp",
         "order_delivered_customer_date"]
    ]
)

New invalid dates introduced during conversion:
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Delivered orders with missing customer delivery date:


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT


In [95]:
print(
    "Approval before purchase:",
    (
        orders_clean["order_approved_at"]
        < orders_clean["order_purchase_timestamp"]
    ).sum()
)

print(
    "Carrier date before purchase:",
    (
        orders_clean["order_delivered_carrier_date"]
        < orders_clean["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before purchase:",
    (
        orders_clean["order_delivered_customer_date"]
        < orders_clean["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before carrier date:",
    (
        orders_clean["order_delivered_customer_date"]
        < orders_clean["order_delivered_carrier_date"]
    ).sum()
)

Approval before purchase: 0
Carrier date before purchase: 166
Customer delivery before purchase: 0
Customer delivery before carrier date: 23


In [96]:
# Save cleaned copy; raw CSV remains unchanged
orders_clean.to_csv(
    INTERIM_FOLDER / "orders_clean.csv",
    index=False
)

# Document date anomalies
anomaly_report = pd.DataFrame({
    "Issue": [
        "Carrier date before purchase date",
        "Customer delivery date before carrier date",
        "Delivered orders missing customer delivery date"
    ],
    "Count": [
        (
            orders_clean["order_delivered_carrier_date"]
            < orders_clean["order_purchase_timestamp"]
        ).sum(),
        (
            orders_clean["order_delivered_customer_date"]
            < orders_clean["order_delivered_carrier_date"]
        ).sum(),
        (
            (orders_clean["order_status"] == "delivered")
            & orders_clean["order_delivered_customer_date"].isna()
        ).sum()
    ],
    "Action": [
        "Retain and investigate",
        "Retain and investigate",
        "Retain; exclude from delivery-time calculation"
    ]
})

anomaly_report.to_csv(
    REPORT_FOLDER / "orders_date_anomalies.csv",
    index=False
)

print("Orders file saved:", (INTERIM_FOLDER / "orders_clean.csv").exists())
print("Rows:", len(orders_clean))
print("Missing cells retained:", orders_clean.isna().sum().sum())
print("\nAnomaly report:")
display(anomaly_report)

Orders file saved: True
Rows: 99441
Missing cells retained: 4908

Anomaly report:


,Issue,Count,Action
0,Carrier date before purchase date,166,Retain and investigate
1,Customer delivery date before carrier date,23,Retain and investigate
2,Delivered orders missing customer delivery date,8,Retain; exclude from delivery-time calculation


In [97]:
products = pd.read_csv(
    RAW_FOLDER / "olist_products_dataset.csv"
)

print("Total rows:", len(products))

print("\nMissing values per column:")
print(products.isna().sum())

print("\nExact duplicate rows:", products.duplicated().sum())

print(
    "Duplicate product_id:",
    products["product_id"].duplicated().sum()
)

print("\nData types:")
print(products.dtypes)

Total rows: 32951

Missing values per column:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Exact duplicate rows: 0
Duplicate product_id: 0

Data types:
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [98]:
# Products with missing category
missing_category = products[
    products["product_category_name"].isna()
]

print("Products with missing category:", len(missing_category))

print("\nMissing values in these products:")
print(missing_category.isna().sum())

print("\nSample products with missing category:")
display(missing_category.head(5))

# Products with missing physical measurements
measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_measurements = products[
    products[measurement_columns].isna().any(axis=1)
]

print("\nProducts with missing measurements:", len(missing_measurements))
display(missing_measurements)

Products with missing category: 610

Missing values in these products:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

Sample products with missing category:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0



Products with missing measurements: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
metadata_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

# Chaaron fields ek saath missing hain
all_metadata_missing = products[metadata_columns].isna().all(axis=1)

# Kam-se-kam ek metadata field missing hai
any_metadata_missing = products[metadata_columns].isna().any(axis=1)

print(
    "All 4 metadata fields missing:",
    all_metadata_missing.sum()
)

print(
    "Only some metadata fields missing:",
    (any_metadata_missing & ~all_metadata_missing).sum()
)

print(
    "Physical measurements missing:",
    products[
        ["product_weight_g", "product_length_cm",
         "product_height_cm", "product_width_cm"]
    ].isna().any(axis=1).sum()
)

All 4 metadata fields missing: 610
Only some metadata fields missing: 0
Physical measurements missing: 2


In [100]:
products_clean = products.copy()

# Missing category ko clearly mark karo
products_clean["product_category_name"] = (
    products_clean["product_category_name"].fillna("Unknown")
)

# Cleaned copy save karo
products_clean.to_csv(
    INTERIM_FOLDER / "products_clean.csv",
    index=False
)

# Remaining missing values ki report
product_missing_report = pd.DataFrame({
    "Column": products_clean.columns,
    "Missing_Values": products_clean.isna().sum().values
})

product_missing_report = product_missing_report[
    product_missing_report["Missing_Values"] > 0
]

product_missing_report.to_csv(
    REPORT_FOLDER / "products_remaining_missing_values.csv",
    index=False
)

print("Products file saved:", (INTERIM_FOLDER / "products_clean.csv").exists())
print("Rows:", len(products_clean))
print("Unknown categories:", (products_clean["product_category_name"] == "Unknown").sum())
print("\nRemaining missing values:")
display(product_missing_report)

Products file saved: True
Rows: 32951
Unknown categories: 610

Remaining missing values:


,Column,Missing_Values
2,product_name_lenght,610
3,product_description_lenght,610
4,product_photos_qty,610
5,product_weight_g,2
6,product_length_cm,2
7,product_height_cm,2
8,product_width_cm,2


In [101]:
sellers = pd.read_csv(
    RAW_FOLDER / "olist_sellers_dataset.csv"
)

print("Total rows:", len(sellers))

print("\nMissing values:")
print(sellers.isna().sum())

print("\nExact duplicate rows:", sellers.duplicated().sum())

print(
    "Duplicate seller_id:",
    sellers["seller_id"].duplicated().sum()
)

print("\nData types:")
print(sellers.dtypes)

Total rows: 3095

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Exact duplicate rows: 0
Duplicate seller_id: 0

Data types:
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object


In [102]:
sellers_clean = sellers.copy()

sellers_clean.to_csv(
    INTERIM_FOLDER / "sellers_clean.csv",
    index=False
)

print("Sellers file saved:",
      (INTERIM_FOLDER / "sellers_clean.csv").exists())

print("Rows:", len(sellers_clean))
print("Missing cells:", sellers_clean.isna().sum().sum())
print("Duplicate rows:", sellers_clean.duplicated().sum())

Sellers file saved: True
Rows: 3095
Missing cells: 0
Duplicate rows: 0


In [103]:
category_translation = pd.read_csv(
    RAW_FOLDER / "product_category_name_translation.csv"
)

print("Total rows:", len(category_translation))

print("\nMissing values:")
print(category_translation.isna().sum())

print("\nExact duplicate rows:",
      category_translation.duplicated().sum())

print("\nDuplicate Portuguese categories:",
      category_translation["product_category_name"].duplicated().sum())

print("\nDuplicate English categories:",
      category_translation["product_category_name_english"].duplicated().sum())

print("\nData types:")
print(category_translation.dtypes)

display(category_translation.head())

Total rows: 71

Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64

Exact duplicate rows: 0

Duplicate Portuguese categories: 0

Duplicate English categories: 0

Data types:
product_category_name            object
product_category_name_english    object
dtype: object


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [104]:
category_translation_clean = category_translation.copy()

category_translation_clean.to_csv(
    INTERIM_FOLDER / "category_translation_clean.csv",
    index=False
)

print(
    "File saved:",
    (INTERIM_FOLDER / "category_translation_clean.csv").exists()
)
print("Rows:", len(category_translation_clean))
print(
    "Missing cells:",
    category_translation_clean.isna().sum().sum()
)
print(
    "Duplicate rows:",
    category_translation_clean.duplicated().sum()
)

File saved: True
Rows: 71
Missing cells: 0
Duplicate rows: 0


In [105]:
expected_files = [
    "customers_clean.csv",
    "geolocation_clean.csv",
    "order_items_clean.csv",
    "payments_clean.csv",
    "reviews_clean.csv",
    "orders_clean.csv",
    "products_clean.csv",
    "sellers_clean.csv",
    "category_translation_clean.csv"
]

verification_results = []

for filename in expected_files:
    filepath = INTERIM_FOLDER / filename

    if not filepath.exists():
        verification_results.append({
            "Dataset": filename,
            "Status": "FILE MISSING",
            "Rows": None,
            "Columns": None,
            "Missing Cells": None,
            "Duplicate Rows": None
        })
        continue

    df_check = pd.read_csv(filepath)

    verification_results.append({
        "Dataset": filename,
        "Status": "Saved",
        "Rows": len(df_check),
        "Columns": len(df_check.columns),
        "Missing Cells": int(df_check.isna().sum().sum()),
        "Duplicate Rows": int(df_check.duplicated().sum())
    })

verification_df = pd.DataFrame(verification_results)

display(verification_df)

,Dataset,Status,Rows,Columns,Missing Cells,Duplicate Rows
0,customers_clean.csv,Saved,99441,5,0,0
1,geolocation_clean.csv,Saved,738332,5,0,0
2,order_items_clean.csv,Saved,112650,7,0,0
3,payments_clean.csv,Saved,103886,5,0,0
4,reviews_clean.csv,Saved,99224,7,145903,0
5,orders_clean.csv,Saved,99441,8,4908,0
6,products_clean.csv,Saved,32951,9,1838,0
7,sellers_clean.csv,Saved,3095,4,0,0
8,category_translation_clean.csv,Saved,71,2,0,0


In [106]:
reviews_clean = pd.read_csv(
    RAW_FOLDER / "olist_order_reviews_dataset.csv"
)

comment_columns = [
    "review_comment_title",
    "review_comment_message"
]

for col in comment_columns:
    reviews_clean[col] = reviews_clean[col].fillna("Not provided")

for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col],
        errors="coerce"
    )

reviews_clean.to_csv(
    INTERIM_FOLDER / "reviews_clean.csv",
    index=False
)

# Saved CSV ko dobara read karke verify karo
reviews_verify = pd.read_csv(
    INTERIM_FOLDER / "reviews_clean.csv"
)

print("Rows:", len(reviews_verify))
print("Missing cells:", reviews_verify.isna().sum().sum())
print("Duplicate rows:", reviews_verify.duplicated().sum())
print(
    "Not provided in comment titles:",
    (reviews_verify["review_comment_title"] == "Not provided").sum()
)
print(
    "Not provided in comment messages:",
    (reviews_verify["review_comment_message"] == "Not provided").sum()
)

Rows: 99224
Missing cells: 0
Duplicate rows: 0
Not provided in comment titles: 87656
Not provided in comment messages: 58247


In [107]:
verification_results = []

for filepath in sorted(INTERIM_FOLDER.glob("*_clean.csv")):
    df_check = pd.read_csv(filepath)

    verification_results.append({
        "Dataset": filepath.name,
        "Rows": len(df_check),
        "Columns": len(df_check.columns),
        "Missing Cells": int(df_check.isna().sum().sum()),
        "Duplicate Rows": int(df_check.duplicated().sum())
    })

final_verification = pd.DataFrame(verification_results)

print(final_verification.to_string(index=False))
print("\nTotal cleaned CSV files:", len(final_verification))

                       Dataset   Rows  Columns  Missing Cells  Duplicate Rows
category_translation_clean.csv     71        2              0               0
           customers_clean.csv  99441        5              0               0
         geolocation_clean.csv 738332        5              0               0
         order_items_clean.csv 112650        7              0               0
              orders_clean.csv  99441        8           4908               0
            payments_clean.csv 103886        5              0               0
            products_clean.csv  32951        9           1838               0
             reviews_clean.csv  99224        7              0               0
             sellers_clean.csv   3095        4              0               0

Total cleaned CSV files: 9


In [108]:

cleaning_log = pd.DataFrame([
    {
        "Dataset": "Customers",
        "Cleaning_Action": "No correction required",
        "Details": "No missing values or duplicate rows"
    },
    {
        "Dataset": "Geolocation",
        "Cleaning_Action": "Removed exact duplicate rows",
        "Details": "261831 duplicates removed; distinct coordinates retained"
    },
    {
        "Dataset": "Order Items",
        "Cleaning_Action": "Validated and saved",
        "Details": "No missing values, duplicates, or invalid dates"
    },
    {
        "Dataset": "Payments",
        "Cleaning_Action": "Validated and saved",
        "Details": "Zero payments and unusual installment values retained for investigation"
    },
    {
        "Dataset": "Reviews",
        "Cleaning_Action": "Handled missing comments",
        "Details": "Missing comments replaced with Not provided; repeated review IDs retained"
    },
    {
        "Dataset": "Orders",
        "Cleaning_Action": "Converted dates and documented anomalies",
        "Details": "Missing dates retained; 166 carrier-date and 23 delivery-date sequence anomalies documented"
    },
    {
        "Dataset": "Products",
        "Cleaning_Action": "Handled missing categories",
        "Details": "610 missing categories marked Unknown; numerical missing values retained"
    },
    {
        "Dataset": "Sellers",
        "Cleaning_Action": "No correction required",
        "Details": "No missing values or duplicate rows"
    },
    {
        "Dataset": "Category Translation",
        "Cleaning_Action": "No correction required",
        "Details": "No missing values or duplicate rows"
    }
])

cleaning_log.to_csv(
    REPORT_FOLDER / "data_cleaning_log.csv",
    index=False
)

print("Cleaning log saved:",
      (REPORT_FOLDER / "data_cleaning_log.csv").exists())

display(cleaning_log)

Cleaning log saved: True


,Dataset,Cleaning_Action,Details
0,Customers,No correction required,No missing values or duplicate rows
1,Geolocation,Removed exact duplicate rows,261831 duplicates removed; distinct coordinate...
2,Order Items,Validated and saved,"No missing values, duplicates, or invalid dates"
3,Payments,Validated and saved,Zero payments and unusual installment values r...
4,Reviews,Handled missing comments,Missing comments replaced with Not provided; r...
5,Orders,Converted dates and documented anomalies,Missing dates retained; 166 carrier-date and 2...
6,Products,Handled missing categories,610 missing categories marked Unknown; numeric...
7,Sellers,No correction required,No missing values or duplicate rows
8,Category Translation,No correction required,No missing values or duplicate rows


In [109]:
final_audit_rows = []
missing_detail_rows = []

for filepath in sorted(INTERIM_FOLDER.glob("*_clean.csv")):
    df_check = pd.read_csv(filepath)

    final_audit_rows.append({
        "Dataset": filepath.name,
        "Rows": len(df_check),
        "Columns": len(df_check.columns),
        "Missing_Cells": int(df_check.isna().sum().sum()),
        "Exact_Duplicate_Rows": int(df_check.duplicated().sum())
    })

    missing_by_column = df_check.isna().sum()

    for column, count in missing_by_column.items():
        if count > 0:
            missing_detail_rows.append({
                "Dataset": filepath.name,
                "Column": column,
                "Missing_Values": int(count)
            })

final_audit_df = pd.DataFrame(final_audit_rows)
missing_details_df = pd.DataFrame(
    missing_detail_rows,
    columns=["Dataset", "Column", "Missing_Values"]
)

# Reports save karo
final_audit_df.to_csv(
    REPORT_FOLDER / "final_data_quality_summary.csv",
    index=False
)

missing_details_df.to_csv(
    REPORT_FOLDER / "remaining_missing_values.csv",
    index=False
)

print("FINAL DATA QUALITY SUMMARY")
display(final_audit_df)

print("\nREMAINING MISSING VALUES — COLUMN-WISE")
display(missing_details_df)

print("\nTotal cleaned files:", len(final_audit_df))
print("Summary report saved:", 
      (REPORT_FOLDER / "final_data_quality_summary.csv").exists())
print("Missing values report saved:",
      (REPORT_FOLDER / "remaining_missing_values.csv").exists())

FINAL DATA QUALITY SUMMARY


,Dataset,Rows,Columns,Missing_Cells,Exact_Duplicate_Rows
0,category_translation_clean.csv,71,2,0,0
1,customers_clean.csv,99441,5,0,0
2,geolocation_clean.csv,738332,5,0,0
3,order_items_clean.csv,112650,7,0,0
4,orders_clean.csv,99441,8,4908,0
5,payments_clean.csv,103886,5,0,0
6,products_clean.csv,32951,9,1838,0
7,reviews_clean.csv,99224,7,0,0
8,sellers_clean.csv,3095,4,0,0



REMAINING MISSING VALUES — COLUMN-WISE


,Dataset,Column,Missing_Values
0,orders_clean.csv,order_approved_at,160
1,orders_clean.csv,order_delivered_carrier_date,1783
2,orders_clean.csv,order_delivered_customer_date,2965
3,products_clean.csv,product_name_lenght,610
4,products_clean.csv,product_description_lenght,610
5,products_clean.csv,product_photos_qty,610
6,products_clean.csv,product_weight_g,2
7,products_clean.csv,product_length_cm,2
8,products_clean.csv,product_height_cm,2
9,products_clean.csv,product_width_cm,2



Total cleaned files: 9
Summary report saved: True
Missing values report saved: True


In [110]:
orders_check = pd.read_csv(
    INTERIM_FOLDER / "orders_clean.csv",
    usecols=["order_id"]
)

items_check = pd.read_csv(
    INTERIM_FOLDER / "order_items_clean.csv",
    usecols=["order_id"]
)

unmatched_items = items_check[
    ~items_check["order_id"].isin(orders_check["order_id"])
]

print("Total order item rows:", len(items_check))
print("Unmatched order item rows:", len(unmatched_items))
print(
    "Unique unmatched order IDs:",
    unmatched_items["order_id"].nunique()
)

Total order item rows: 112650
Unmatched order item rows: 0
Unique unmatched order IDs: 0


In [111]:
orders_check = pd.read_csv(
    INTERIM_FOLDER / "orders_clean.csv",
    usecols=["order_id"]
)

payments_check = pd.read_csv(
    INTERIM_FOLDER / "payments_clean.csv",
    usecols=["order_id"]
)

unmatched_payments = payments_check[
    ~payments_check["order_id"].isin(orders_check["order_id"])
]

print("Total payment rows:", len(payments_check))
print("Unmatched payment rows:", len(unmatched_payments))
print(
    "Unique unmatched order IDs:",
    unmatched_payments["order_id"].nunique()
)

Total payment rows: 103886
Unmatched payment rows: 0
Unique unmatched order IDs: 0


In [112]:
reviews_check = pd.read_csv(
    INTERIM_FOLDER / "reviews_clean.csv",
    usecols=["order_id"]
)

orders_check = pd.read_csv(
    INTERIM_FOLDER / "orders_clean.csv",
    usecols=["order_id"]
)

unmatched_reviews = reviews_check[
    ~reviews_check["order_id"].isin(orders_check["order_id"])
]

print("Total review rows:", len(reviews_check))
print("Unmatched review rows:", len(unmatched_reviews))
print(
    "Unique unmatched order IDs:",
    unmatched_reviews["order_id"].nunique()
)

Total review rows: 99224
Unmatched review rows: 0
Unique unmatched order IDs: 0


In [113]:
orders_check = pd.read_csv(
    INTERIM_FOLDER / "orders_clean.csv",
    usecols=["customer_id"]
)

customers_check = pd.read_csv(
    INTERIM_FOLDER / "customers_clean.csv",
    usecols=["customer_id"]
)

unmatched_orders = orders_check[
    ~orders_check["customer_id"].isin(
        customers_check["customer_id"]
    )
]

print("Total orders:", len(orders_check))
print("Unmatched order rows:", len(unmatched_orders))
print(
    "Unique unmatched customer IDs:",
    unmatched_orders["customer_id"].nunique()
)

Total orders: 99441
Unmatched order rows: 0
Unique unmatched customer IDs: 0


In [114]:
items_check = pd.read_csv(
    INTERIM_FOLDER / "order_items_clean.csv",
    usecols=["product_id"]
)

products_check = pd.read_csv(
    INTERIM_FOLDER / "products_clean.csv",
    usecols=["product_id"]
)

unmatched_items = items_check[
    ~items_check["product_id"].isin(
        products_check["product_id"]
    )
]

print("Total order item rows:", len(items_check))
print("Unmatched item rows:", len(unmatched_items))
print(
    "Unique unmatched product IDs:",
    unmatched_items["product_id"].nunique()
)


Total order item rows: 112650
Unmatched item rows: 0
Unique unmatched product IDs: 0


In [115]:
items_check = pd.read_csv(
    INTERIM_FOLDER / "order_items_clean.csv",
    usecols=["seller_id"]
)

sellers_check = pd.read_csv(
    INTERIM_FOLDER / "sellers_clean.csv",
    usecols=["seller_id"]
)

unmatched_sellers = items_check[
    ~items_check["seller_id"].isin(
        sellers_check["seller_id"]
    )
]

print("Total order item rows:", len(items_check))
print("Unmatched item rows:", len(unmatched_sellers))
print(
    "Unique unmatched seller IDs:",
    unmatched_sellers["seller_id"].nunique()
)

Total order item rows: 112650
Unmatched item rows: 0
Unique unmatched seller IDs: 0


In [116]:
relationship_report = pd.DataFrame([
    {
        "Relationship": "Order Items -> Orders",
        "Key": "order_id",
        "Unmatched_Rows": 0,
        "Status": "Passed"
    },
    {
        "Relationship": "Payments -> Orders",
        "Key": "order_id",
        "Unmatched_Rows": 0,
        "Status": "Passed"
    },
    {
        "Relationship": "Reviews -> Orders",
        "Key": "order_id",
        "Unmatched_Rows": 0,
        "Status": "Passed"
    },
    {
        "Relationship": "Orders -> Customers",
        "Key": "customer_id",
        "Unmatched_Rows": 0,
        "Status": "Passed"
    },
    {
        "Relationship": "Order Items -> Products",
        "Key": "product_id",
        "Unmatched_Rows": 0,
        "Status": "Passed"
    },
    {
        "Relationship": "Order Items -> Sellers",
        "Key": "seller_id",
        "Unmatched_Rows": 0,
        "Status": "Passed"
    }
])

relationship_report.to_csv(
    REPORT_FOLDER / "relationship_validation.csv",
    index=False
)

print("Report saved:",
      (REPORT_FOLDER / "relationship_validation.csv").exists())

display(relationship_report)

Report saved: True


,Relationship,Key,Unmatched_Rows,Status
0,Order Items -> Orders,order_id,0,Passed
1,Payments -> Orders,order_id,0,Passed
2,Reviews -> Orders,order_id,0,Passed
3,Orders -> Customers,customer_id,0,Passed
4,Order Items -> Products,product_id,0,Passed
5,Order Items -> Sellers,seller_id,0,Passed
